# RAG and Agent Evaluation

So far, we evaluated retrieval. We checked whether search returns the document that should answer the question.

That is only the first step. A complete application still needs to produce a final answer. For RAG, this means checking the generated answer. For agents, it also means looking at the tool calls the model made before producing the answer.

RAG evaluation checks the whole flow together.

This includes:

- search
- prompt
- LLM

If the final answer is bad, the problem can come from any of these steps. The search might retrieve the wrong document, the prompt might omit important context, or the LLM might ignore the context.

Now, we'll evaluate:

- RAG answers with an LLM judge
- Agent answers and tool-call trajectories

## LLM as a judge
For RAG and agent evaluation, we compare the generated answer with the original answer. The generated answer won't use the same words as the original. It's a generative model, so the phrasing will be different even when the meaning is the same.

This is why we use another LLM to do the comparison. We show the judge the question, the original answer, and the generated answer. Then we ask it to decide if they are semantically equivalent.

This approach is called LLM-as-a-judge. The evaluating LLM is the judge. It classifies each answer as good or bad and explains its reasoning. Asking the judge to explain why it made a decision generally produces better classifications than asking for just the verdict.

# Generating RAG Answers

Now we evaluate the full RAG pipeline. For each generated question, we run RAG and save the answer produced by the LLM. Later, we'll compare this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our questions came from FAQ records. For each question, we know which original answer it came from.

## Loading the data

Load the ground truth questions:

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the FAQ documents and the search index:

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table for the original FAQ documents:

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

We'll use this lookup table to find the original answer for each ground truth question.

## Running RAG

Import the usual things first:

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

We will use `RAGWithUsage` from the evaluation utilities. It subclasses `RAGBase` from module 1, so it has the same rag method.

It stores token usage after each LLM call. Then we can calculate the total cost later.

It also uses the search boosts we selected during search tuning: `question=1.0`, `answer=2.0`, and `section=0.1`.

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, `RAGBase` searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. We save the answer so the next lesson can judge it.

Run RAG for one question:

In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still start now. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

Check the cost of this call:

In [7]:
assistant.total_cost()

0.000597

Get the original answer from the document ID:

In [8]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Now save both answers in one record:

In [9]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course — is it too late to join, or can I still start now?',
 'answer_llm': 'Yes, you can still start now. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

## Processing all questions

Create a function that processes one ground truth record:

In [10]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

Test it on one record:

In [11]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — is it too late to join, or can I still start now?',
 'answer_llm': 'Yes — you can still start now. You can join the course anytime, and if you want a certificate, just make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Before running the full batch, reset the usage we collected while testing:

In [12]:
assistant.reset_usage()

This calls the LLM once per ground truth question, so it can take some time. Let's process the questions in parallel and track progress.

Import the parallel processing helper from the same utility file:

In [13]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

Run RAG for all ground truth questions:

In [14]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/720 [00:00<?, ?it/s]

`generate_rag_answer` returns one answer record for each question.

Collect the answer records:

In [15]:
answers = []

for answer_record in results:
    answers.append(answer_record)

Calculate the total cost:

In [16]:
assistant.total_cost()

0.7718700000000006

Save the answers:

In [17]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)